In [2]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate
)
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)


In [3]:
topic = "Python"
difficulty = "初学者"

# 难以维护，容易出错
prompt_str = f"你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。"
print(f"提示词:{prompt_str}")

response = model.invoke(prompt_str)
print(f"AI 回复:{response.content[:100]}...\n")

print("-------------------------------")
template = PromptTemplate.from_template(
    "你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。"
)

print(template)
print(f"模板:{template.template}")
print(f"变量:{template.input_variables}")

# 使用模板生成提示词
prompt = template.format(difficulty=difficulty, topic=topic)
print(f"生成的提示词:{prompt}")

response = model.invoke(prompt)
print(f"AI 回复:{response.content[:100]}...\n")


提示词:你是一个初学者级别的编程导师。请用简单易懂的语言解释Python。
AI 回复:Python是一种计算机语言，允许你指示计算机执行任务。想象你正在与一位非常听话的朋友交谈，你要求他们为你做某事。就像给朋友一份清单，告诉他们按照特定的顺序做某些事情一样，你可以给计算机一份指令清单，...

-------------------------------
input_variables=['difficulty', 'topic'] input_types={} partial_variables={} template='你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。'
模板:你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。
变量:['difficulty', 'topic']
生成的提示词:你是一个初学者级别的编程导师。请用简单易懂的语言解释Python。
AI 回复:Python 是一种非常容易学习的编程语言，我很高兴能帮助你入门。以下是简要介绍：

**什么是Python？**
Python是一种计算机语言，允许您编写指令，这些指令可以被计算机理解和执行。它就像...



In [4]:
# 方法 1:使用 from_template(最简单）
print("\n【方法 1:from_template(推荐）】")

template1 = PromptTemplate.from_template(
    "将以下文本翻译成{language}:\n{text}"
)

prompt1 = template1.format(language="法语", text="Hello, how are you?")
print(f"生成的提示词:\n{prompt1}\n")

response1 = model.invoke(prompt1)
print(f"AI 回复:{response1.content}\n")

# 方法 2:显式指定变量(更严格）
print("【方法 2:显式指定变量】")
template2 = PromptTemplate(
    input_variables=["product", "feature"],
    template="为{product}写一句广告语，重点突出{feature}特点。"
)

prompt2 = template2.format(product="智能手表", feature="超长续航")
print(f"生成的提示词:\n{prompt2}\n")

response2 = model.invoke(prompt2)
print(f"AI 回复:{response2.content}\n")

# 方法 3:使用 invoke(直接生成消息）
print("【方法 3:使用 invoke(更方便）】")
template3 = PromptTemplate.from_template(
    "写一首关于{theme}的{style}风格的诗，不超过4行。"
)

# invoke 直接返回格式化后的值
prompt_value = template3.invoke({"theme": "春天", "style": "现代"})
print(f"生成的提示词:\n{prompt_value.text}\n")



【方法 1:from_template(推荐）】
生成的提示词:
将以下文本翻译成法语:
Hello, how are you?

AI 回复:Bonjour, comment vas-tu ?

【方法 2:显式指定变量】
生成的提示词:
为智能手表写一句广告语，重点突出超长续航特点。

AI 回复:“解放你的生活，不再频繁充电——我们的智能手表带来无与伦比的长电池续航时间，让你可以不间断地追踪、连接和探索长达一周。”

【方法 3:使用 invoke(更方便）】
生成的提示词:
写一首关于春天的现代风格的诗，不超过4行。



In [5]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}，擅长{expertise}。"),
    ("user", "请帮我{task}")
])

print(f"模板变量：{chat_template.input_variables}")

# 格式化模板
messages = chat_template.format_messages(
    role="Python 导师",
    expertise="用简单的方式解释复杂概念",
    task="解释什么是列表推导式"
)

print("\n生成的消息：")
for msg in messages:
    print(f"  {msg.type}: {msg.content}")

response = model.invoke(messages)
print(f"\nAI 回复：{response.content[:150]}...\n")

# 方法 2：使用字符串简写（最简洁）
print("【方法 2：字符串简写】")

simple_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手"),
    ("user", "{question}")
])

messages = simple_template.format_messages(question="什么是机器学习？")
response = model.invoke(messages)
print(f"AI 回复：{response.content[:100]}...\n")

模板变量：['expertise', 'role', 'task']

生成的消息：
  system: 你是一个Python 导师，擅长用简单的方式解释复杂概念。
  human: 请帮我解释什么是列表推导式

AI 回复：**列表推导式：一种简洁的创建列表的方式**

列表推导式是一种强大的Python工具，允许您使用简洁的语法创建新列表。它是一种以更少的代码行创建列表的方法。

**什么是列表推导式？**
-...

【方法 2：字符串简写】
AI 回复：机器学习是人工智能的一个分支，它使计算机能够从数据中自动学习和改进，而无需通过明确的编程来实现这一点。它涉及训练算法，使其能够根据数据模式和关系做出预测或决策。

机器学习的基本思想是提供大量数据样本...



In [6]:
# 创建包含对话历史的模板
template = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}。{instruction}"),
    ("user", "{question1}"),
    ("assistant", "{answer1}"),
    ("user", "{question2}")
])

print(template.input_types)
print(type(template))
print(template.input_variables)


# 填充模板
messages = template.format_messages(
    role="Python 专家",
    instruction="回答要简洁、准确",
    question1="什么是列表？",
    answer1="列表是 Python 中的有序可变集合，用方括号 [] 表示。",
    question2="它和元组有什么区别？"  # 基于上下文的问题
)


for i, msg in enumerate(messages, 1):
    content_preview = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
    print(f"  {i}. [{msg.type}] {content_preview}")
    # print(msg.response_metadata)
    # print(msg.content[0])

response = model.invoke(messages)
print(f"\nAI 回复：{response.content}\n")

{}
<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
['answer1', 'instruction', 'question1', 'question2', 'role']
  1. [system] 你是一个Python 专家。回答要简洁、准确
  2. [human] 什么是列表？
  3. [ai] 列表是 Python 中的有序可变集合，用方括号 [] 表示。
  4. [human] 它和元组有什么区别？

AI 回复：列表是可变的，元组是不可变的。列表用方括号[]，元组用圆括号()。



In [7]:
system_template = SystemMessagePromptTemplate.from_template(
    "你是一个{profession}，你的特长是{specialty}。"
)

human_template = HumanMessagePromptTemplate.from_template(
    "关于{topic}，我想知道{question}"
)

# 组合成 ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages([
    system_template,
    human_template
])


print(f"总变量：{chat_template.input_variables}\n")

# 使用模板
messages = chat_template.format_messages(
    profession="数据科学家",
    specialty="用数据讲故事",
    topic="数据可视化",
    question="如何选择合适的图表类型？"
)

response = model.invoke(messages)
print(f"AI 回复：{response.content[:200]}...\n")

总变量：['profession', 'question', 'specialty', 'topic']

AI 回复：选择合适的图表类型是有效数据可视化的关键方面。以下是一些帮助您为数据选择合适图表类型的指导原则：

1. **定义目的**：在选择图表类型之前，确定您要传达的信息。您想要展示趋势、比较、关系还是分布？
2. **了解数据**：考虑数据的类型（定量、分类、时间序列等）和范围。数据是连续的、离散的，还是混合的？
3. **选择合适的图表类型**：
	* **条形图**：比较不同类别之间的值。
	* *...



In [8]:
original_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}，你的目标用户是{audience}。"),
    ("user", "请{task}")
])

print(f"原始模板变量：{original_template.input_variables}\n")

# 部分填充：固定 role 和 audience
partially_filled = original_template.partial(
    role="科技博客作者",
    audience="程序员"
)

print(f"部分填充后的变量：{partially_filled.input_variables}\n")

# 现在只需要提供 task
messages1 = partially_filled.format_messages(
    task="写一篇关于 Python 装饰器的文章开头"
)
print("----------------")
print(len(messages1))
for msg in messages1:
    print(msg.content)
print("----------------")

response1 = model.invoke(messages1)
print(f"文章 1：{response1.content[:150]}...\n")
messages1.append(response1)
print("----------------")
print(len(messages1))
for msg in messages1:
    print(msg.content)
print("----------------")


# 复用模板，不同的 task
messages2 = partially_filled.format_messages(
    task="写一篇关于异步编程的文章开头"
)
print("----------------")
print(len(messages2))
for msg in messages2:
    print(msg.content)
print("----------------")
response2 = model.invoke(messages2)
print(f"文章 2：{response2.content[:150]}...\n")

原始模板变量：['audience', 'role', 'task']

部分填充后的变量：['task']

----------------
2
你是一个科技博客作者，你的目标用户是程序员。
请写一篇关于 Python 装饰器的文章开头
----------------
文章 1：**Python装饰器：给代码添加超能力**

作为一名Python开发人员，您可能曾经遇到过需要为现有函数或方法添加新功能的情况，而不必修改其底层实现。这可能是一个具有挑战性和容易出错的任务，尤其是当您处理复杂的代码库时。这就是Python装饰器的用处——一个强大且灵活的工具，可以帮助您以优雅而非...

----------------
3
你是一个科技博客作者，你的目标用户是程序员。
请写一篇关于 Python 装饰器的文章开头
**Python装饰器：给代码添加超能力**

作为一名Python开发人员，您可能曾经遇到过需要为现有函数或方法添加新功能的情况，而不必修改其底层实现。这可能是一个具有挑战性和容易出错的任务，尤其是当您处理复杂的代码库时。这就是Python装饰器的用处——一个强大且灵活的工具，可以帮助您以优雅而非侵入性的方式修改和扩展代码的行为。在这篇文章中，我们将深入探讨Python装饰器的世界，探索它们的基础知识、用例和最佳实践，包括如何使用它们来记录函数、实现授权和缓存结果。通过掌握装饰器，您将能够编写更干净、更高效、更易于维护的代码，使您成为更强大的Python开发人员。
----------------
----------------
2
你是一个科技博客作者，你的目标用户是程序员。
请写一篇关于异步编程的文章开头
----------------
文章 2：**解锁异步编程的力量：提高性能和可扩展性的指南**

作为开发人员，我们不断地被要求编写能够高效、可靠地处理大量任务的代码。然而，传统的顺序编程方法可能会导致性能瓶颈、可扩展性问题和令人沮丧的用户体验。在这种情况下，异步编程出现了，它通过允许代码以非阻塞方式执行多个任务，从而彻底改变了编码的方式。...



In [13]:
template = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}"),
    ("user", "{input}")
])

# 使用 | 运算符创建链
chain = template | model

# 直接调用链
response = chain.invoke({
    "role": "幽默的程序员",
    "input": "解释什么是bug"
})

print(response.content[:100])



臭虫。这些讨厌的、让人头疼的、让你想从头上拔出头发的……（深呼吸）……代码中的错误。是的，就是这样。

想象一下，你精心制作了一份美味的食谱（也就是你的代码），但不知怎么的，一个多余的成分（也就是 b
